# 01. 지연과 최단 경로의 기초
목표: 논문 §2–3의 전송·전파·큐 지연 및 Dijkstra의 세 가중치를 구분한다. Python 3.10+ 표준 라이브러리만 필요하다. 위에서부터 실행하거나 `python verify_notebooks.py`를 사용한다.

**Toy reproduction**: 4개 노드의 가상 그래프다. 실제 궤도·RF/FSO 링크 예산을 재현하지 않는다. 수치는 교육용이며 논문 결과가 아니다. [원문](https://doi.org/10.5281/zenodo.13885645).

In [ ]:
import math
import heapq
C = 299_792_458.0  # m/s. km를 그대로 넣으면 1000배 오류가 난다.
def delay_s(packet_bytes, rate_bps, distance_km, queue_s=0.0):
    values = (packet_bytes, rate_bps, distance_km, queue_s)
    if not all(math.isfinite(x) for x in values):
        raise ValueError('finite inputs required')
    if packet_bytes <= 0 or rate_bps <= 0 or distance_km < 0 or queue_s < 0:
        raise ValueError('invalid physical inputs')
    transmission = packet_bytes * 8 / rate_bps
    propagation = distance_km * 1000 / C
    return {'queue': queue_s, 'transmission': transmission, 'propagation': propagation,
            'total': queue_s + transmission + propagation}
parts = delay_s(1500, 10e6, 1000)
assert math.isclose(parts['transmission'], 0.0012)
assert math.isclose(parts['total'], 0.00453564095198152)
print('1500 byte / 10 Mbit/s / 1000 km:', {k: round(v * 1000, 6) for k,v in parts.items()}, 'ms')
assert 8 * (8 - 1) == 56  # 방향을 구분한 gateway 쌍


## 같은 그래프, 다른 최적화 목표
A–D는 한 홉이지만 느리다. A–B–D는 빠른 링크지만 멀다. A–C–D는 가장 짧은 거리다. 거리·속도는 임의 값이며 물리적으로 배치된 실제 위성이 아니다. 모든 간선은 양방향, 큐는 우선 0이다. Dijkstra는 경로 전체의 가중치 합을 최소화한다.

In [ ]:
edges = [('A','D',100,1e6), ('A','B',1000,100e6), ('B','D',1000,100e6),
         ('A','C',10,5e6), ('C','D',10,5e6)]
graph = {n: {} for n in 'ABCD'}
for u,v,km,bps in edges:
    graph[u][v] = graph[v][u] = {'km': km, 'bps': bps}
def shortest_path(g, start, destination, metric):
    if start not in g or destination not in g:
        raise ValueError('unknown node')
    costs = {'hop': lambda e: 1, 'range': lambda e: e['km'],
             'rate': lambda e: 1 / e['bps']}
    weight = costs[metric]
    heap, best = [(0.0, start, [start])], {start: 0.0}
    while heap:
        cost, node, path = heapq.heappop(heap)
        if cost > best[node]:
            continue
        if node == destination:
            return cost, path
        for nxt, link in g[node].items():
            if link['bps'] <= 0:  # R=0인 링크는 나누지 않고 연결 불가로 취급
                continue
            w = weight(link)
            if not math.isfinite(w) or w < 0:
                raise ValueError('Dijkstra requires finite nonnegative weights')
            candidate = cost + w
            if candidate < best.get(nxt, math.inf):
                best[nxt] = candidate
                heapq.heappush(heap, (candidate, nxt, path + [nxt]))
    raise ValueError('no route')
expected = {'hop': ['A','D'], 'range': ['A','C','D'], 'rate': ['A','B','D']}
for metric in expected:
    cost, path = shortest_path(graph, 'A', 'D', metric)
    assert path == expected[metric]
    latency = sum(delay_s(1500, graph[u][v]['bps'], graph[u][v]['km'])['total']
                  for u,v in zip(path, path[1:]))
    print(metric, '->'.join(path), 'no-queue latency(ms)=', round(latency*1000, 4))
assert shortest_path(graph, 'A', 'A', 'hop') == (0.0, ['A'])
try:
    shortest_path({'A': {}, 'D': {}}, 'A', 'D', 'hop')
    raise AssertionError('disconnected graph accepted')
except ValueError:
    pass
try:
    delay_s(1500, 0, 10)
    raise AssertionError('zero rate accepted')
except ValueError:
    pass


## 결과 해석과 확장
세 정책이 다른 경로를 고르면 정상이다. rate 최적화는 큐·전파를 포함한 E2E 최적화와 다르다. 전체 지연을 계산할 때 각 홉의 queue를 임의로 동일 값으로 더하는 모델도 실제 경쟁을 재현하지 못한다. 다음 실습에서 큐를 직접 시뮬레이션한다.

과제: 패킷을 100배 크게 만들었을 때 rate 경로와 range 경로의 순위가 어떻게 바뀌는가? bytes→bits 변환을 생략하면 어떤 오차가 생기는가? R=0인 간선과 목적지 단절 조건을 추가해 검사하라.